In [ ]:
pip install sktime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 32.0 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.8/159.8 kB 6.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [ ]:
pip install optuna

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gc
import pandas as pd # Added for results table
from sklearn.linear_model import RidgeClassifierCV, RidgeClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay, f1_score
from sklearn.utils import resample
from sklearn.model_selection import learning_curve # train_test_split removed for Optuna's internal use as per user request
from sktime.transformations.panel.rocket import MiniRocketMultivariate
import optuna # New import
from scipy.stats import kurtosis, skew # New imports for statistical features
from sklearn.preprocessing import StandardScaler # New import for statistical feature scaling
from sklearn.pipeline import make_pipeline # New import for pipeline

# ==========================================
# 1. CARREGAMENTO E PREPARAÇÃO DOS DADOS
# ==========================================
print("1. Carregando dados do UTD-MHAD (formato .npz com train/val/test splits)...")
data_path = "/content/drive/MyDrive/2025/Estudos/Datasets/utd-mhad/utd_mhad_v2.npz"
dados = np.load(data_path)

# Carrega os splits individuais
X_train_orig = dados['X_train']
y_train_orig = dados['y_train']
X_val_orig = dados['X_val']
y_val_orig = dados['y_val']
X_test = dados['X_test']
y_test = dados['y_test']

# MiniRocketMultivariate espera o formato (n_instances, n_channels, n_timepoints)
# O arquivo .npz tem (n_instances, n_timepoints, n_channels), entao transpomos.
# Shape original do .npz: (N, 125, 66) -> Transposto para (N, 66, 125)
# Keeping original splits separate as requested for Optuna tuning
X_train_raw = np.transpose(X_train_orig, (0, 2, 1))
y_train_raw = y_train_orig
X_val_raw = np.transpose(X_val_orig, (0, 2, 1))
y_val_raw = y_val_orig
X_test_raw = np.transpose(X_test, (0, 2, 1))
y_test_raw = y_test

print(f"Shape X_train (original, transposto) para MiniRocket: {X_train_raw.shape} | 66 Canais")
print(f"Shape y_train (original): {y_train_raw.shape}")
print(f"Shape X_val (original, transposto) para MiniRocket:   {X_val_raw.shape} | 66 Canais")
print(f"Shape y_val (original): {y_val_raw.shape}")
print(f"Shape X_test (transposto) para MiniRocket:  {X_test_raw.shape} | 66 Canais")
print(f"Shape y_test:  {y_test_raw.shape}")

del dados, X_train_orig, y_train_orig, X_val_orig, y_val_orig
gc.collect()

# Function to transform data in batches (from original notebook)
def transform_in_batches(model, X, batch_size=2000):
    n_samples = X.shape[0]
    features = []
    for i in range(0, n_samples, batch_size):
        fim = min(i + batch_size, n_samples)
        feat_batch = model.transform(X[i:fim]).astype(np.float32)
        features.append(feat_batch)
    return np.vstack(features)

# New function for statistical feature extraction
def extract_statistical_features(X):
    # X shape is (N, C, T)
    mean_features = np.mean(X, axis=2)
    std_features = np.std(X, axis=2)
    min_features = np.min(X, axis=2)
    max_features = np.max(X, axis=2)
    kurt_features = kurtosis(X, axis=2)
    skew_features = skew(X, axis=2)
    return np.concatenate([mean_features, std_features, min_features, max_features, kurt_features, skew_features], axis=1)


# ==========================================
# Cenários de Treinamento e Avaliação
# ==========================================

def run_scenario(scenario_name, X_train_raw_input, y_train_raw_input, X_val_raw_input, y_val_raw_input, X_test_raw_input, y_test_raw_input, include_statistical_features, use_optuna):
    print(f"\n{'='*70}")
    print(f"Executando {scenario_name}")
    print(f"{'='*70}\n")

    # Step 2: Feature Engineering (MiniRocket is always included for these scenarios)
    print("2. Extraindo features com MiniRocket...")
    minirocket = MiniRocketMultivariate()
    # Fit MiniRocket strictly on X_train to avoid data leakage
    minirocket.fit(X_train_raw_input)

    # Transform all splits using the fitted MiniRocket
    X_train_minirocket = transform_in_batches(minirocket, X_train_raw_input)
    X_val_minirocket = transform_in_batches(minirocket, X_val_raw_input)
    X_test_minirocket = transform_in_batches(minirocket, X_test_raw_input)
    print(f"   Shape X_train (MiniRocket): {X_train_minirocket.shape}")
    print(f"   Shape X_val (MiniRocket):   {X_val_minirocket.shape}")
    print(f"   Shape X_test (MiniRocket):  {X_test_minirocket.shape}")

    X_train_processed_for_tuning = X_train_minirocket
    X_val_processed_for_tuning = X_val_minirocket
    X_test_processed = X_test_minirocket

    if include_statistical_features:
        print("2.1. Extraindo features estatísticas adicionais (média, std, min, max, kurtose, assimetria)...")
        X_train_stats = extract_statistical_features(X_train_raw_input)
        X_val_stats = extract_statistical_features(X_val_raw_input)
        X_test_stats = extract_statistical_features(X_test_raw_input)

        # Escalonamento será feito pelo make_pipeline final
        # scaler = StandardScaler()
        # X_train_stats_scaled = scaler.fit_transform(X_train_stats)
        # X_val_stats_scaled = scaler.transform(X_val_stats)
        # X_test_stats_scaled = scaler.transform(X_test_stats)

        print(f"   Shape X_train (Estatísticas brutas): {X_train_stats.shape}")
        print(f"   Shape X_val (Estatísticas brutas):   {X_val_stats.shape}")
        print(f"   Shape X_test (Estatísticas brutas):  {X_test_stats.shape}")

        # Concatenate MiniRocket and raw statistical features for each split
        X_train_processed_for_tuning = np.concatenate([X_train_minirocket, X_train_stats], axis=1)
        X_val_processed_for_tuning = np.concatenate([X_val_minirocket, X_val_stats], axis=1)
        X_test_processed = np.concatenate([X_test_minirocket, X_test_stats], axis=1)
        print(f"   Shape X_train (MiniRocket + Estatísticas brutas): {X_train_processed_for_tuning.shape}")
        print(f"   Shape X_val (MiniRocket + Estatísticas brutas):   {X_val_processed_for_tuning.shape}")
        print(f"   Shape X_test (MiniRocket + Estatísticas brutas):  {X_test_processed.shape}")
    else:
        print("2.1. Sem features estatísticas adicionais (apenas MiniRocket).")

    gc.collect() # Clean up memory after FE

    # Prepare training data for the classifier (only using X_train, no concatenation with X_val)
    X_train_for_clf_final = X_train_processed_for_tuning
    y_train_for_clf_final = y_train_raw_input

    # 3. HYPERPARAMETER TUNING / CLASSIFIER SETUP
    melhor_alpha = None
    melhor_solver = 'auto'
    if use_optuna:
        print("3. Otimizando hiperparâmetros com Optuna (usando X_train e X_val)...")
        print(f"   Optuna Training Shape: {X_train_processed_for_tuning.shape}")
        print(f"   Optuna Validation Shape: {X_val_processed_for_tuning.shape}")

        def objective(trial):
            alpha = trial.suggest_float('alpha', 1e-3, 1e3, log=True)
            solver = trial.suggest_categorical('solver', ["auto", "svd", "cholesky", "lsqr", "sparse_cg"])
            classifier = make_pipeline(StandardScaler(), RidgeClassifier(alpha=alpha, solver=solver, class_weight='balanced', random_state=42))
            classifier.fit(X_train_processed_for_tuning, y_train_raw_input)
            predictions = classifier.predict(X_val_processed_for_tuning)
            return f1_score(y_val_raw_input, predictions, average='macro')

        study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
        study.optimize(objective, n_trials=50, show_progress_bar=True)
        melhor_alpha = study.best_params['alpha']
        melhor_solver = study.best_params['solver']
        print(f"   Melhor Alpha (Optuna): {melhor_alpha:.4f}")
        print(f"   Melhor Solver (Optuna): {melhor_solver}")

    else:
        print("3. Configurando classificador Ridge com Pesos de Classe (usando RidgeClassifierCV no X_train)...")
        clf_cv = make_pipeline(StandardScaler(), RidgeClassifierCV(alphas=np.logspace(-3, 3, 10), class_weight='balanced'))
        clf_cv.fit(X_train_for_clf_final, y_train_for_clf_final)
        melhor_alpha = clf_cv.named_steps['ridgeclassifiercv'].alpha_
        print(f"   Melhor Alpha (RidgeClassifierCV): {melhor_alpha:.4f}")

    # Final training and evaluation
    print("\n   Treinamento final e Avaliação...")
    clf_final = make_pipeline(StandardScaler(), RidgeClassifier(alpha=melhor_alpha, solver=melhor_solver, class_weight='balanced', random_state=42))
    clf_final.fit(X_train_for_clf_final, y_train_for_clf_final)
    previsoes = clf_final.predict(X_test_processed)

    # Verificação da importância das features
    if include_statistical_features:
        print("\n   --- Análise de Importância das Features ---")
        n_minirocket_feats = X_train_minirocket.shape[1]
        coefs = np.abs(clf_final.named_steps['ridgeclassifier'].coef_)
        mean_coefs = np.mean(coefs, axis=0)

        importancia_minirocket = np.mean(mean_coefs[:n_minirocket_feats])
        importancia_stats = np.mean(mean_coefs[n_minirocket_feats:])

        print(f"   Importância Média - MiniRocket: {importancia_minirocket:.6f}")
        print(f"   Importância Média - Estatísticas: {importancia_stats:.6f}")
        print("   -------------------------------------------\n")

    accuracy = accuracy_score(y_test_raw_input, previsoes)
    macro_f1 = f1_score(y_test_raw_input, previsoes, average='macro')

    print(f"\nMelhor Alpha Final: {melhor_alpha:.4f} | Solver: {melhor_solver}")
    print("Acurácia Base:", accuracy)
    print("Macro F1-Score:", macro_f1)
    print("\nRelatório de Classificação:\n", classification_report(y_test_raw_input, previsoes))

    # Matriz de Confusão
    fig, ax = plt.subplots(figsize=(12, 10))
    ConfusionMatrixDisplay.from_predictions(y_test_raw_input, previsoes, cmap='Blues', ax=ax, normalize='true')
    plt.title(f"Matriz de Confusão - {scenario_name}")
    plt.xticks(rotation=45)
    plt.show()

    # 4. BOOTSTRAP VALIDATION
    print("\n4. Iniciando Validação Bootstrap (F1-Score Macro)...")
    n_iterations = 30
    bootstrap_results = []

    for i in range(n_iterations):
        X_resampled, y_resampled = resample(X_train_for_clf_final, y_train_for_clf_final, random_state=i)
        it_clf = make_pipeline(StandardScaler(), RidgeClassifier(alpha=melhor_alpha, solver=melhor_solver, class_weight='balanced', random_state=42))
        it_clf.fit(X_resampled, y_resampled)
        score = f1_score(y_test_raw_input, it_clf.predict(X_test_processed), average='macro')
        bootstrap_results.append(score)
        del X_resampled, y_resampled
        gc.collect()

    bootstrap_mean = np.mean(bootstrap_results)
    bootstrap_std = np.std(bootstrap_results)
    print(f"Média Bootstrap: {bootstrap_mean:.4f} | Desvio Padrão: {bootstrap_std:.4f}")

    # 5. LEARNING CURVE
    print("\n5. Gerando Curva de Aprendizado...")
    train_sizes, train_scores, test_scores = learning_curve(
        make_pipeline(StandardScaler(), RidgeClassifier(alpha=melhor_alpha, solver=melhor_solver, class_weight='balanced', random_state=42)),
        X_train_for_clf_final, y_train_for_clf_final,
        cv=3, n_jobs=1, scoring='f1_macro',
        train_sizes=np.linspace(0.1, 1.0, 5),
        random_state=42
    )

    plt.figure(figsize=(10, 6))
    plt.plot(train_sizes, np.mean(train_scores, axis=1), 'o-', label="Treino (F1)")
    plt.plot(train_sizes, np.mean(test_scores, axis=1), 's-', label="Validação (F1)")
    plt.title(f"Curva de Aprendizado - {scenario_name}")
    plt.xlabel("Amostras de Treino")
    plt.ylabel("F1-Score Macro")
    plt.legend()
    plt.grid(True)
    plt.show()

    print(f"\nFinalizado {scenario_name}\n")
    return {
        "Scenario": scenario_name,
        "Accuracy": accuracy,
        "Macro F1-Score": macro_f1,
        "Bootstrap Mean F1": bootstrap_mean,
        "Bootstrap Std F1": bootstrap_std
    }


# Define the four scenarios
scenarios = [
    {"name": "Cenário 1: Apenas MiniRocket, Sem Optuna", "include_statistical_features": False, "use_optuna": False},
    {"name": "Cenário 2: Apenas MiniRocket, Com Optuna", "include_statistical_features": False, "use_optuna": True},
    {"name": "Cenário 3: MiniRocket + Estatísticas, Sem Optuna", "include_statistical_features": True, "use_optuna": False},
    {"name": "Cenário 4: MiniRocket + Estatísticas, Com Optuna", "include_statistical_features": True, "use_optuna": True},
]

all_scenario_results = []

# Run each scenario
for scenario in scenarios:
    results = run_scenario(
        scenario["name"],
        X_train_raw,
        y_train_raw,
        X_val_raw,
        y_val_raw,
        X_test_raw,
        y_test_raw,
        scenario["include_statistical_features"],
        scenario["use_optuna"]
    )
    all_scenario_results.append(results)

# Display comparative table
results_df = pd.DataFrame(all_scenario_results)
print("\n" + "="*70)
print("Tabela Comparativa de Resultados (F1-Score Macro e Acurácia)")
print("="*70)
display(results_df)

# Plot comparative histogram/bar chart
plt.figure(figsize=(14, 7))
bar_width = 0.35
index = np.arange(len(results_df))

plt.bar(index, results_df['Macro F1-Score'], bar_width, label='Macro F1-Score')
plt.bar(index + bar_width, results_df['Accuracy'], bar_width, label='Acurácia')

plt.xlabel('Cenário')
plt.ylabel('Score')
plt.title('Comparativo de Performance por Cenário')
plt.xticks(index + bar_width / 2, results_df['Scenario'], rotation=45, ha='right')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()